# 25 - Analyze source online threshold refinement

This compares the causal online `U >= 0.03` source rollout with the exact historical unrefined
source identity. Every success rate uses the whole matched episode set. Historical always-refine
is shown only as context. The online gate is evaluated independently at Euler steps 3 and 4; this
is not the old post-hoc whole-episode threshold policy.

## 1. Setup

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Fetch exact matched identities

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pnp.config import Method
from pnp.diversity import (DIVERSITY_FIXED_REFINEMENT_THRESHOLD, DIVERSITY_PAIR_KEYS,
    SOURCE_THRESHOLD_REFINEMENT_EXPERIMENT)
from pnp.experiments import PRO_EXPANDED_EXPERIMENT
from pnp.store import SupabaseStore

THRESHOLD = DIVERSITY_FIXED_REFINEMENT_THRESHOLD
OUTPUT = Path("source_threshold_refinement_outputs")
OUTPUT.mkdir(exist_ok=True)
store = SupabaseStore()

online = pd.DataFrame(store.fetch_all(
    "rollouts", "*", configure=lambda query: query.eq(
        "experiment", SOURCE_THRESHOLD_REFINEMENT_EXPERIMENT).eq(
            "method", Method.THRESHOLD_REFINEMENT), order_by=("rollout_id",)))
source = pd.DataFrame(store.fetch_all(
    "rollouts", "*", configure=lambda query: query.eq(
        "experiment", PRO_EXPANDED_EXPERIMENT), order_by=("rollout_id",)))
assert len(online), "No online threshold-refinement rollouts found"
online = online[online.status.eq("completed")].copy()
source = source[
    source.status.eq("completed") & source.pnp_k.eq(5) &
    source.pnp_step_indices.apply(lambda value: tuple(value or []) == (3, 4))].copy()
baseline = source[source.method.eq(Method.UNCERTAINTY)].copy()
always = source[
    source.method.eq(Method.REFINEMENT) & ~source.refine_average.fillna(False).astype(bool)].copy()
for name, frame in (("online", online), ("baseline", baseline), ("always", always)):
    assert not frame.duplicated(DIVERSITY_PAIR_KEYS).any(), f"duplicate {name} identities"

paired = (online[DIVERSITY_PAIR_KEYS + ["rollout_id", "success", "n_pnp_activations",
                                        "n_corrections_applied", "gate_fire_rate"]]
          .rename(columns={"rollout_id": "online_rollout_id",
                           "success": "online_success"})
          .merge(baseline[DIVERSITY_PAIR_KEYS + ["success"]].rename(
              columns={"success": "baseline_success"}),
              on=DIVERSITY_PAIR_KEYS, validate="one_to_one")
          .merge(always[DIVERSITY_PAIR_KEYS + ["success"]].rename(
              columns={"success": "always_refine_success"}),
              on=DIVERSITY_PAIR_KEYS, how="left", validate="one_to_one"))
assert len(paired) == len(online), (
    f"Only {len(paired)}/{len(online)} online rows match the historical baseline")
for column in ("online_success", "baseline_success"):
    paired[column] = paired[column].astype(bool)
print({"completed_online_rollouts": len(online), "matched_identities": len(paired),
       "suites": paired.suite.nunique(), "threshold": THRESHOLD})

## 3. Whole-cohort result and gate usage

In [ ]:
def summarize(group):
    baseline_values = group.baseline_success.to_numpy(bool)
    online_values = group.online_success.to_numpy(bool)
    considered = int(group.n_pnp_activations.fillna(0).sum())
    applied = int(group.n_corrections_applied.fillna(0).sum())
    return pd.Series({
        "episodes": len(group),
        "unrefined_baseline_sr_pct": 100 * baseline_values.mean(),
        "online_threshold_sr_pct": 100 * online_values.mean(),
        "online_minus_baseline_pp": 100 * (online_values.mean() - baseline_values.mean()),
        "failure_to_success": int((~baseline_values & online_values).sum()),
        "success_to_failure": int((baseline_values & ~online_values).sum()),
        "probe_steps_considered": considered,
        "refinement_steps_applied": applied,
        "gate_fire_rate_pct": 100 * applied / max(considered, 1),
        "historical_always_refine_sr_pct": 100 * group.always_refine_success.mean(),
    })

overall = summarize(paired).to_frame().T
by_suite = pd.DataFrame([
    {"suite": suite, **summarize(group).to_dict()}
    for suite, group in paired.groupby("suite", sort=True)])
print("Primary result: all matched episodes remain in the SR denominator")
display(overall)
display(by_suite[["suite", "episodes", "unrefined_baseline_sr_pct",
                  "online_threshold_sr_pct", "online_minus_baseline_pp",
                  "gate_fire_rate_pct"]])
paired.to_csv(OUTPUT / "matched_episodes.csv", index=False)
overall.to_csv(OUTPUT / "overall.csv", index=False)
by_suite.to_csv(OUTPUT / "by_suite.csv", index=False)

## 4. Success-rate change and gate diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(by_suite)); width = .38
axes[0].bar(x - width/2, by_suite.unrefined_baseline_sr_pct, width,
            label="unrefined source", color="#4C78A8")
axes[0].bar(x + width/2, by_suite.online_threshold_sr_pct, width,
            label="online U-gated refinement", color="#F58518")
axes[0].set_xticks(x, by_suite.suite.str.removeprefix("libero_"), rotation=40, ha="right")
axes[0].set(ylabel="Success rate (%)", ylim=(0, 105),
            title="Online threshold refinement vs matched source baseline")
axes[0].legend()

colors = np.where(by_suite.online_minus_baseline_pp >= 0, "#54A24B", "#E45756")
axes[1].bar(x, by_suite.online_minus_baseline_pp, color=colors)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_xticks(x, by_suite.suite.str.removeprefix("libero_"), rotation=40, ha="right")
axes[1].set(ylabel="SR change (percentage points)",
            title="Whole-cohort SR change by suite")
fig.tight_layout()
fig.savefig(OUTPUT / "source_online_threshold_refinement.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(100 * paired.gate_fire_rate.dropna(), bins=15, color="#72B7B2", edgecolor="white")
ax.set(xlabel="Episode gate-fire rate (% of probed Euler steps)", ylabel="Episodes",
       title="How often did U >= 0.03 trigger refinement?")
fig.tight_layout()
fig.savefig(OUTPUT / "source_online_threshold_gate_rate.png", dpi=180, bbox_inches="tight")
plt.show()